In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("customer behavior.csv", sep=";")

print("DATA AWAL:")
display(df)

DATA AWAL:


,Customer ID,Gender,Age,City,Membership Type,Total Spend,Items Purchased,Average Rating,Discount Applied,Days Since Last Purchase,Satisfaction Level
0,101,Female,29,New York,Gold,11202,14,46,True,25,Satisfied
1,102,Male,34,Los Angeles,Silver,7805,11,41,False,18,Neutral
2,103,Female,43,Chicago,Bronze,51075,9,34,True,42,Unsatisfied
3,104,Male,30,San Francisco,Gold,14803,19,47,False,12,Satisfied
4,105,Male,27,Miami,Silver,7204,13,4,True,55,Unsatisfied
...,...,...,...,...,...,...,...,...,...,...,...
345,446,Male,32,Miami,Silver,6603,10,38,True,42,Unsatisfied
346,447,Female,36,Houston,Bronze,4705,8,3,False,27,Neutral
347,448,Female,30,New York,Gold,11908,16,45,True,28,Satisfied
348,449,Male,34,Los Angeles,Silver,7802,11,42,False,21,Neutral


In [3]:
df.duplicated().sum()

df = df.drop_duplicates()

In [4]:
#karena skala rating 1-50 dan ada yang hanya satu digit saja dan polanya pasti dibawah angka 5 berarti pola nol dibelakng hilang yang cukup masuk akal

df["Average Rating"] = df["Average Rating"].apply(
    lambda x: x * 10 if x < 10 else x
)

In [5]:
df.isnull().sum()


Customer ID                 0
Gender                      0
Age                         0
City                        0
Membership Type             0
Total Spend                 0
Items Purchased             0
Average Rating              0
Discount Applied            0
Days Since Last Purchase    0
Satisfaction Level          2
dtype: int64

In [6]:
df = df.fillna({
    "Gender": "Unknown",
    "City": "Unknown",
    "Membership Type": "Unknown",
    "Average Rating": df["Average Rating"].median(),
    "Satisfaction Level": "Unknown"
})


In [7]:
df.isnull().sum()

Customer ID                 0
Gender                      0
Age                         0
City                        0
Membership Type             0
Total Spend                 0
Items Purchased             0
Average Rating              0
Discount Applied            0
Days Since Last Purchase    0
Satisfaction Level          0
dtype: int64

In [8]:
Q1 = df["Total Spend"].quantile(0.25)
Q3 = df["Total Spend"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["Outlier_TotalSpend"] = df["Total Spend"].apply(
    lambda x: "YES" if x < lower_bound or x > upper_bound else "NO"
)

print("\nDETEKSI OUTLIER TOTAL SPEND:")
display(df[["Customer ID", "Total Spend", "Outlier_TotalSpend"]])


DETEKSI OUTLIER TOTAL SPEND:


,Customer ID,Total Spend,Outlier_TotalSpend
0,101,11202,NO
1,102,7805,NO
2,103,51075,YES
3,104,14803,NO
4,105,7204,NO
...,...,...,...
345,446,6603,NO
346,447,4705,NO
347,448,11908,NO
348,449,7802,NO


In [9]:
outlier_count = df[df["Outlier_TotalSpend"] == "YES"].shape[0]
print("Jumlah Outlier:", outlier_count)

Jumlah Outlier: 58


In [10]:
df = df.drop(columns=["Outlier_TotalSpend"])


bukan outlier secara bisnis konteks, karena jumlah spend yang masih wajar dan masih mungkin dalam sebuah pembelian, jadi yg saya ubah hanya rating, dan mengisi null yang kosong pada tingkat kepuasan secara unknown, tidak ada duplikat

In [11]:
# Kolom patokan duplikat
cols_to_check = ['Age', 'City', 'Items Purchased', 'Satisfaction Level', 'Membership Type','Average Rating']

# Hapus duplikat (menyimpan baris pertama dari setiap kombinasi)
df_cleaned = df.drop_duplicates(subset=cols_to_check, keep='first')

print("Data setelah duplikat dihapus:")
print(df_cleaned)

Data setelah duplikat dihapus:
     Customer ID  Gender  Age           City Membership Type  Total Spend  \
0            101  Female   29       New York            Gold        11202   
1            102    Male   34    Los Angeles          Silver         7805   
2            103  Female   43        Chicago          Bronze        51075   
3            104    Male   30  San Francisco            Gold        14803   
4            105    Male   27          Miami          Silver         7204   
5            106  Female   37        Houston          Bronze         4408   
6            107  Female   31       New York            Gold        11506   
7            108    Male   35    Los Angeles          Silver         8009   
8            109  Female   41        Chicago          Bronze        49525   
9            110    Male   28  San Francisco            Gold        15201   
10           111    Male   32          Miami          Silver         6903   
11           112  Female   36        Houston 

In [12]:
df.drop_duplicates(subset=cols_to_check, keep='first', inplace=True)
df.to_csv("data.csv", index=False)
